# 03 - ML Time-Series Models

Models evaluated:
- Linear Regression
- XGBoost (if installed)

Using lag, rolling, and calendar features with recursive forecasting.

In [4]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ARTIFACTS_DIR = BASE_DIR / 'artifacts'
DAILY_PATH = ARTIFACTS_DIR / 'daily_series.csv'

HORIZON = 30
N_SPLITS = 3

daily = pd.read_csv(DAILY_PATH, parse_dates=['date']).set_index('date')
print('XGBoost available:', XGBOOST_AVAILABLE)
daily.head()

XGBoost available: True


,daily_revenue,daily_orders
date,,
2024-02-10,31740.30,122
2024-02-11,37996.92,141
2024-02-12,35297.50,139
2024-02-13,40122.00,150
2024-02-14,35806.81,152


In [5]:
def safe_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.where(np.abs(y_true) < 1e-9, np.nan, np.abs(y_true))
    return np.nanmean(np.abs((y_true - y_pred) / denom)) * 100


def metric_row(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    return {'MAE': mae, 'MSE': mse, 'RMSE': np.sqrt(mse), 'MAPE': safe_mape(y_true, y_pred)}


def build_rolling_splits(series, horizon=30, n_splits=3):
    first_train_end = len(series) - horizon * n_splits
    if first_train_end <= 35:
        raise ValueError('Not enough data for requested rolling setup.')
    out = []
    for i in range(n_splits):
        train_end = first_train_end + i * horizon
        train = series.iloc[:train_end]
        test = series.iloc[train_end:train_end + horizon]
        out.append((train, test))
    return out


def make_supervised_frame(series, lags=(1,2,3,7,14,28), roll_windows=(7,14)):
    d = pd.DataFrame({'y': series.astype(float)})
    for lag in lags:
        d[f'lag_{lag}'] = d['y'].shift(lag)
    for w in roll_windows:
        d[f'roll_mean_{w}'] = d['y'].shift(1).rolling(w).mean()
        d[f'roll_std_{w}'] = d['y'].shift(1).rolling(w).std()

    idx = d.index
    d['dow'] = idx.dayofweek
    d['is_weekend'] = (idx.dayofweek >= 5).astype(int)
    d['month'] = idx.month
    d['quarter'] = idx.quarter

    return d.dropna()


def make_feature_row(history, dt, lags=(1,2,3,7,14,28), roll_windows=(7,14)):
    row = {}
    for lag in lags:
        row[f'lag_{lag}'] = history.iloc[-lag] if len(history) >= lag else np.nan
    for w in roll_windows:
        vals = history.iloc[-w:] if len(history) >= w else history
        row[f'roll_mean_{w}'] = vals.mean() if len(vals) else np.nan
        row[f'roll_std_{w}'] = vals.std() if len(vals) > 1 else 0.0

    row['dow'] = dt.dayofweek
    row['is_weekend'] = int(dt.dayofweek >= 5)
    row['month'] = dt.month
    row['quarter'] = ((dt.month - 1) // 3) + 1

    return pd.DataFrame([row], index=[dt]).fillna(0.0)


def recursive_forecast(model, train_series, horizon):
    history = train_series.copy()
    future_idx = pd.date_range(start=history.index[-1] + pd.Timedelta(days=1), periods=horizon, freq='D')
    preds = []
    for dt in future_idx:
        x_row = make_feature_row(history, dt)
        p = float(model.predict(x_row)[0])
        p = max(0.0, p)
        preds.append(p)
        history.loc[dt] = p
    return pd.Series(preds, index=future_idx)

In [6]:
def run_ml_model(model_name, train, horizon, xgb_params=None):
    sup = make_supervised_frame(train)
    X = sup.drop(columns=['y'])
    y = sup['y']

    if model_name == 'LinearRegression':
        model = LinearRegression()
    elif model_name == 'XGBoost':
        if not XGBOOST_AVAILABLE:
            raise ImportError('XGBoost not installed')
        params = xgb_params or {
            'n_estimators': 300,
            'learning_rate': 0.05,
            'max_depth': 4,
            'subsample': 0.9,
            'colsample_bytree': 0.9,
            'random_state': 42,
            'objective': 'reg:squarederror'
        }
        model = XGBRegressor(**params)
    else:
        raise ValueError(model_name)

    model.fit(X, y)
    pred = recursive_forecast(model, train, horizon)
    return pred


def tune_xgboost_params(series, seasonal_horizon=30):
    if not XGBOOST_AVAILABLE:
        return None

    split_point = len(series) - seasonal_horizon
    if split_point <= 45:
        return {
            'n_estimators': 300,
            'learning_rate': 0.05,
            'max_depth': 4,
            'subsample': 0.9,
            'colsample_bytree': 0.9,
            'random_state': 42,
            'objective': 'reg:squarederror'
        }

    train = series.iloc[:split_point]
    valid = series.iloc[split_point:]

    grid = [
        {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 3, 'subsample': 0.9, 'colsample_bytree': 0.9, 'random_state': 42, 'objective': 'reg:squarederror'},
        {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 4, 'subsample': 0.9, 'colsample_bytree': 0.9, 'random_state': 42, 'objective': 'reg:squarederror'},
        {'n_estimators': 500, 'learning_rate': 0.03, 'max_depth': 4, 'subsample': 0.9, 'colsample_bytree': 0.8, 'random_state': 42, 'objective': 'reg:squarederror'},
        {'n_estimators': 400, 'learning_rate': 0.07, 'max_depth': 5, 'subsample': 0.8, 'colsample_bytree': 0.9, 'random_state': 42, 'objective': 'reg:squarederror'}
    ]

    best_params = None
    best_rmse = np.inf
    for params in grid:
        try:
            pred = run_ml_model('XGBoost', train, len(valid), xgb_params=params)
            pred = pd.Series(pred.values, index=valid.index)
            rmse = np.sqrt(mean_squared_error(valid.values, pred.values))
            if rmse < best_rmse:
                best_rmse = rmse
                best_params = params
        except Exception:
            pass

    if best_params is None:
        best_params = {
            'n_estimators': 300,
            'learning_rate': 0.05,
            'max_depth': 4,
            'subsample': 0.9,
            'colsample_bytree': 0.9,
            'random_state': 42,
            'objective': 'reg:squarederror'
        }
    return best_params


targets = ['daily_revenue', 'daily_orders']
models = ['LinearRegression'] + (['XGBoost'] if XGBOOST_AVAILABLE else [])

target_tuning = {}
rows = []
for target in targets:
    series = daily[target].astype(float)
    splits = build_rolling_splits(series, horizon=HORIZON, n_splits=N_SPLITS)

    tuned_xgb = tune_xgboost_params(series, seasonal_horizon=HORIZON) if XGBOOST_AVAILABLE else None
    target_tuning[target] = {'XGBoost': tuned_xgb} if tuned_xgb else {}

    for split_no, (train, test) in enumerate(splits, start=1):
        for model_name in models:
            try:
                pred = run_ml_model(model_name, train, len(test), xgb_params=tuned_xgb)
                pred = pd.Series(pred.values, index=test.index)
                m = metric_row(test.values, pred.values)
                rows.append({
                    'target': target,
                    'split': split_no,
                    'model': model_name,
                    **m
                })
            except Exception as ex:
                rows.append({
                    'target': target,
                    'split': split_no,
                    'model': model_name,
                    'MAE': np.nan,
                    'MSE': np.nan,
                    'RMSE': np.nan,
                    'MAPE': np.nan,
                    'error': str(ex)
                })

ml_cv = pd.DataFrame(rows)
ml_summary = (
    ml_cv
    .groupby(['target', 'model'], as_index=False)[['MAE', 'MSE', 'RMSE', 'MAPE']]
    .mean()
    .sort_values(['target', 'RMSE'])
)

ml_cv.to_csv(ARTIFACTS_DIR / 'ml_cv_metrics.csv', index=False)
ml_summary.to_csv(ARTIFACTS_DIR / 'ml_metrics_summary.csv', index=False)

with open(ARTIFACTS_DIR / 'ml_tuning_params.json', 'w', encoding='utf-8') as f:
    json.dump(target_tuning, f, indent=2)

ml_summary

,target,model,MAE,MSE,RMSE,MAPE
0,daily_orders,LinearRegression,10.127928,1.634950e+02,12.651407,7.654634
1,daily_orders,XGBoost,11.333072,1.985388e+02,13.980754,8.670421
2,daily_revenue,LinearRegression,2833.988901,1.202910e+07,3449.220633,8.436801
3,daily_revenue,XGBoost,3083.345455,1.388638e+07,3718.480522,9.212893
